In [1]:
# ================================================================
# PHASE 3 — CGRS: Curvature-Guided Rank Selection for ViTs
# Research: Curvature-Guided Rank Selection (CGRS)
# USC MS Computer Science (AI) — Spring 2026
#
# ALGORITHM (Global Rank Adjustment):
#   1. Start training with r_init (default r=16)
#   2. Every K steps: compute diagonal Fisher → extract lambda_max
#      (cheap probe: only N_PROBE_BATCHES × batch_size=4 samples)
#   3. If lambda_max > tau: rank too low → increase rank by one step
#   4. If lambda_max < tau: rank may be excessive → decrease rank
#   5. Rank transition:
#        Increase → zero-pad A and B matrices (new dims start at 0)
#        Decrease → SVD truncation of B@A, keep top new_r components
#   6. Clamp rank within [R_MIN, R_MAX] at all times
#   7. Cooldown COOLDOWN steps after any rank change (no thrashing)
#
# COMPARISON PLAN:
#   Fixed LoRA r=6    — Phase 1 result (no retraining needed)
#   Fixed LoRA r=10   — Phase 1 result (no retraining needed)
#   Fixed LoRA r=16   — Phase 1 result (no retraining needed)
#   CGRS tau=0.01     — high flatness target → converges near r=16–30
#   CGRS tau=0.05     — moderate target    → converges near r=8–16
#   CGRS tau=0.10     — low flatness target → converges near r=5–10
#
# PRIMARY METRIC: does CGRS achieve r=16 accuracy at avg_rank < 16?
#
# PREREQUISITES:
#   phase1_results/config.json
#   phase1_results/results.json
#   phase2_results/all_results_complete.json   (tau calibration)
#
# OUTPUTS:
#   phase3_results/phase3_results.json         — all experiment results
#   phase3_results/cgrs_{label}_trajectory.json — rank/lambda per run
# ================================================================

In [2]:
!pip install torch torchvision transformers peft scipy numpy --break-system-packages

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: /apps/spack/2406/apps/linux-rocky8-x86_64_v3/gcc-13.3.0/python-3.12.8-4ph2byz/bin/python -m pip install --upgrade pip


In [3]:
import sys
print(sys.executable)
!python3 -m pip list | grep numpy

/apps/conda/envs/ood-jupyterlab-4.5/bin/python
DEPRECATION: The default format will switch to columns in the future. You can use --format=(legacy|columns) (or define a format=(legacy|columns) in your pip.conf under the [list] section) to disable this warning.
numpy (1.19.5)


In [4]:
# ----------------------------------------------------------------
# CELL 1 — Imports and reproducibility
# Identical seed setup to Phases 1 and 2 for consistent splits.
# ----------------------------------------------------------------
import os
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from transformers import ViTForImageClassification, ViTImageProcessor
from peft import LoraConfig, get_peft_model
from scipy.stats import pearsonr, spearmanr
from collections import defaultdict

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

/home1/hsharma9/.local/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device : cpu


/home1/hsharma9/.local/lib/python3.14/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12090). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [5]:
# ----------------------------------------------------------------
# CELL 2 — Config + load Phase 1/2 results + tau calibration
#
# tau values are calibrated from Phase 2 lambda_max results.
# If phase2 results are not yet available, falls back to defaults
# informed by the preliminary 4-rank study.
#
# CGRS-specific hyperparameters:
#   cgrs_check_every    — steps between Fisher probe + rank checks
#   cgrs_cooldown       — min steps between consecutive rank changes
#   cgrs_probe_batches  — batches for the cheap Fisher probe
#   cgrs_r_init         — starting rank
#   cgrs_r_min/r_max    — hard clamps on dynamic rank
#   cgrs_rank_list      — valid ranks (from Phase 1 sweep)
#   cgrs_tau_values     — sharpness targets to test
# ----------------------------------------------------------------
PHASE1_DIR = "."          # config.json and results.json are here
PHASE2_DIR = "."          # all_results_complete.json would be here too
PHASE3_DIR = "./phase3_results"
CHECKPOINTS_DIR = "./phase1_results"   # lora_r{r}_full.pt and lora_r{r}_adapter/
os.makedirs(PHASE3_DIR, exist_ok=True)

# Load Phase 1 config and results from root
with open("./config.json") as f:
    CONFIG = json.load(f)
with open("./results.json") as f:
    phase1_results = json.load(f)

LORA_RANKS = [3, 5, 6, 10, 12, 16, 30, 52, 64, 80, 100, 128, 150, 200]
CONFIG["lora_ranks"] = LORA_RANKS

# ---- Load Phase 2 curvature results (for tau calibration) ----
p2_complete_path = f"{PHASE2_DIR}/all_results_complete.json"
phase2_available = os.path.exists(p2_complete_path)

if phase2_available:
    with open(p2_complete_path) as f:
        all_results_p2 = json.load(f)
    curvature_data = all_results_p2.get("curvature", {})
    print("Phase 2 results loaded — computing tau from actual lambda_max values.")
    # Build a rank → lambda_max lookup for tau calibration
    lmax_by_rank = {}
    for r in LORA_RANKS:
        key = f"r={r}"
        if key in curvature_data:
            lmax_by_rank[r] = curvature_data[key]["lambda_max"]
    if lmax_by_rank:
        lmax_values = list(lmax_by_rank.values())
        print(f"  lambda_max range: {min(lmax_values):.6f} — {max(lmax_values):.6f}")
        # tau_high targets ~r=16–30 range (25th pctile of lambda_max)
        # tau_mid targets ~r=6–16 range (50th pctile)
        # tau_low targets ~r=3–10 range (75th pctile)
        sorted_lmax = sorted(lmax_values)
        n = len(sorted_lmax)
        tau_high = sorted_lmax[n // 4]      # aggressive flatness
        tau_mid  = sorted_lmax[n // 2]      # moderate
        tau_low  = sorted_lmax[3 * n // 4]  # permissive
        TAU_VALUES = [tau_high, tau_mid, tau_low]
        print(f"  Calibrated tau values: high={tau_high:.6f}, mid={tau_mid:.6f}, low={tau_low:.6f}")
    else:
        print("  Warning: curvature_data empty — using default tau values.")
        TAU_VALUES = [0.01, 0.05, 0.10]
        lmax_by_rank = {}
else:
    print("Phase 2 results not found — using default tau values from preliminary study.")
    print("  Upload phase2_results/all_results_complete.json for calibrated tau.")
    TAU_VALUES   = [0.01, 0.05, 0.10]
    lmax_by_rank = {}
    curvature_data = {}

# ---- CGRS hyperparameters ----
CONFIG["cgrs_check_every"]   = 200    # steps between rank checks
CONFIG["cgrs_cooldown"]      = 100    # min steps after a rank change
CONFIG["cgrs_probe_batches"] = 8      # batches × 4 = 32 samples per probe
CONFIG["cgrs_r_init"]        = 16     # starting rank
CONFIG["cgrs_r_min"]         = 3      # minimum allowed rank
CONFIG["cgrs_r_max"]         = 64     # maximum allowed rank (memory cap)
CONFIG["cgrs_rank_list"]     = [r for r in LORA_RANKS if r <= CONFIG["cgrs_r_max"]]
CONFIG["cgrs_tau_values"]    = TAU_VALUES

# Save updated config
with open(f"{PHASE3_DIR}/config_phase3.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

print(f"\nCGRS config:")
print(f"  r_init          : {CONFIG['cgrs_r_init']}")
print(f"  r_min / r_max   : {CONFIG['cgrs_r_min']} / {CONFIG['cgrs_r_max']}")
print(f"  rank_list       : {CONFIG['cgrs_rank_list']}")
print(f"  check_every     : {CONFIG['cgrs_check_every']} steps")
print(f"  cooldown        : {CONFIG['cgrs_cooldown']} steps")
print(f"  probe_batches   : {CONFIG['cgrs_probe_batches']} (={CONFIG['cgrs_probe_batches']*4} samples)")
print(f"  tau_values      : {[f'{t:.4f}' for t in TAU_VALUES]}")
print(f"\nPhase 1 baseline results:")
for k, v in phase1_results.items():
    p = v.get('trainable_params', 'n/a')
    print(f"  {k:<22} -> Test Acc: {v['test_acc']:.2f}%  Params: {p}")

Phase 2 results not found — using default tau values from preliminary study.
  Upload phase2_results/all_results_complete.json for calibrated tau.

CGRS config:
  r_init          : 16
  r_min / r_max   : 3 / 64
  rank_list       : [3, 5, 6, 10, 12, 16, 30, 52, 64]
  check_every     : 200 steps
  cooldown        : 100 steps
  probe_batches   : 8 (=32 samples)
  tau_values      : ['0.0100', '0.0500', '0.1000']

Phase 1 baseline results:
  Full fine-tune         -> Test Acc: 92.87%  Params: 86567396
  Frozen backbone        -> Test Acc: 86.46%  Params: 76900
  LoRA r=3               -> Test Acc: 84.77%  Params: 110592
  LoRA r=5               -> Test Acc: 87.36%  Params: 184320
  LoRA r=6               -> Test Acc: 87.27%  Params: 221184
  LoRA r=10              -> Test Acc: 89.02%  Params: 368640
  LoRA r=12              -> Test Acc: 89.31%  Params: 442368
  LoRA r=16              -> Test Acc: 89.51%  Params: 589824
  LoRA r=30              -> Test Acc: 90.30%  Params: 1105920
  LoRA r=5

In [6]:
# ----------------------------------------------------------------
# CELL 3 — Dataset and DataLoaders
# Loads processor from local Phase 1 adapter checkpoint instead
# of HuggingFace Hub — no internet required.
# ----------------------------------------------------------------

# Find any saved adapter folder from Phase 1 to load processor from
adapter_path = None
for r in LORA_RANKS:
    candidate = f"./phase1_results/lora_r{r}_adapter"
    if os.path.isdir(candidate):
        adapter_path = candidate
        print(f"Found adapter: {candidate}")
        break

if adapter_path:
    print(f"Loading processor from local adapter: {adapter_path}")
    processor = ViTImageProcessor.from_pretrained(adapter_path, local_files_only=True)
else:
    # Fallback: use hardcoded ImageNet mean/std (same values ViT-Base always uses)
    print("No adapter found — using hardcoded ImageNet normalization.")
    from transformers import ViTImageProcessor
    # ViT-Base/16 always uses these exact values
    IMAGENET_MEAN = [0.5, 0.5, 0.5]
    IMAGENET_STD  = [0.5, 0.5, 0.5]

if adapter_path:
    mean = processor.image_mean
    std  = processor.image_std
else:
    mean = IMAGENET_MEAN
    std  = IMAGENET_STD

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

print(f"  Normalization mean: {mean}")
print(f"  Normalization std : {std}")

print("Loading CIFAR-100...")
full_train   = datasets.CIFAR100(root="./data", train=True,  download=True, transform=transform)
test_dataset = datasets.CIFAR100(root="./data", train=False, download=True, transform=transform)

val_size = len(full_train) - CONFIG["train_size"]
train_dataset, val_dataset = random_split(
    full_train,
    [CONFIG["train_size"], val_size],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(
    train_dataset, batch_size=CONFIG["batch_size"], shuffle=True,
    num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=CONFIG["batch_size"], shuffle=False,
    num_workers=2, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=CONFIG["batch_size"], shuffle=False,
    num_workers=2, pin_memory=True
)
curvature_probe_loader = DataLoader(
    train_dataset, batch_size=4, shuffle=False,
    num_workers=2, pin_memory=True
)

print(f"  Train : {len(train_dataset):,}")
print(f"  Val   : {len(val_dataset):,}")
print(f"  Test  : {len(test_dataset):,}")
print(f"  Probe loader : batch_size=4 | {len(curvature_probe_loader)} batches total")

No adapter found — using hardcoded ImageNet normalization.
  Normalization mean: [0.5, 0.5, 0.5]
  Normalization std : [0.5, 0.5, 0.5]
Loading CIFAR-100...


/home1/hsharma9/.local/lib/python3.14/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


  Train : 45,000
  Val   : 5,000
  Test  : 10,000
  Probe loader : batch_size=4 | 11250 batches total


In [7]:
# ----------------------------------------------------------------
# CELL 4 — Model builders and evaluation
#
# build_lora_model_silent — builds without printing (used during
#   rank transitions so the console stays clean)
# build_lora_model_verbose — prints trainable param count
#   (used for the initial model in each experiment)
# evaluate — shared eval loop (identical to Phase 1)
# ----------------------------------------------------------------
def count_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    return trainable, total


def build_lora_model_silent(r):
    """Build LoRA ViT-Base with rank r. No console output."""
    base = ViTForImageClassification.from_pretrained(
        CONFIG["model_name"],
        num_labels=CONFIG["num_classes"],
        ignore_mismatched_sizes=True,
    )
    cfg = LoraConfig(
        r              = r,
        lora_alpha     = CONFIG["lora_alpha"],
        lora_dropout   = CONFIG["lora_dropout"],
        target_modules = CONFIG["target_modules"],
        bias           = "none",
    )
    return get_peft_model(base, cfg).to(device)


def build_lora_model_verbose(r):
    """Build LoRA ViT-Base with rank r. Prints trainable params."""
    model = build_lora_model_silent(r)
    t, total = count_parameters(model)
    print(f"  LoRA r={r:<4} | Trainable: {t:>10,} / {total:,} ({100*t/total:.3f}%)")
    return model, t


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    criterion  = nn.CrossEntropyLoss()
    loss_sum, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images  = images.to(device, non_blocking=True)
        labels  = labels.to(device, non_blocking=True)
        outputs = model(images)
        loss_sum += criterion(outputs.logits, labels).item()
        correct  += (outputs.logits.argmax(dim=1) == labels).sum().item()
        total    += labels.size(0)
    return loss_sum / len(loader), 100.0 * correct / total


print("Model builders and evaluate() ready.")

Model builders and evaluate() ready.


In [8]:
# ----------------------------------------------------------------
# CELL 5 — Rank transition and Fisher probe utilities
#
# transition_lora_rank:
#   Creates a new LoRA model at new_r, transferring weights from
#   old_model (at old_r). This is the core CGRS mechanic.
#
#   Increase (new_r > old_r):  zero-pad A and B matrices.
#     lora_A: shape (r, in_features)  → pad rows with zeros
#     lora_B: shape (out_features, r) → pad columns with zeros
#   The padded dimensions contribute zero to the output initially
#   and can be learned from the current training step onward.
#
#   Decrease (new_r < old_r): SVD truncation of the composite
#   update matrix W = B @ A. The SVD finds the best rank-new_r
#   approximation: W ≈ U[:, :new_r] * diag(sqrt(S)) @ Vh[:new_r, :].
#   This preserves the maximum variance of the learned update.
#
# probe_fisher:
#   Cheap (N batches × 4 samples) diagonal Fisher probe.
#   Used inside the CGRS loop every K steps.
# ----------------------------------------------------------------
_criterion_probe = nn.CrossEntropyLoss()


def transition_lora_rank(old_model, old_r, new_r):
    """
    Build a new LoRA model at new_r, initialized from old_model (old_r).
    Non-LoRA parameters (frozen base weights, classifier) are copied directly.
    Returns the new model on device.
    """
    # Snapshot old model's parameters to CPU before building new model
    old_params = {name: param.data.clone().cpu()
                  for name, param in old_model.named_parameters()}

    # Build new architecture on CPU first (saves VRAM during transition)
    new_model = build_lora_model_silent(new_r)
    new_model.cpu()

    # ---- Step 1: Copy all non-LoRA parameters (frozen base + classifier) ----
    new_param_dict = dict(new_model.named_parameters())
    for name, old_data in old_params.items():
        if 'lora_A' not in name and 'lora_B' not in name:
            if name in new_param_dict:
                new_param_dict[name].data.copy_(old_data)

    # ---- Step 2: Transfer LoRA matrices with rank adjustment ----
    # Collect A/B pairs by their shared module base path
    # PEFT names: ...attention.query.lora_A.default.weight
    #             ...attention.query.lora_B.default.weight
    lora_bases = {}
    for name, data in old_params.items():
        if 'lora_A' in name:
            base = name[:name.index('lora_A')]
            lora_bases.setdefault(base, {})
            lora_bases[base]['A']      = data
            lora_bases[base]['name_A'] = name
        elif 'lora_B' in name:
            base = name[:name.index('lora_B')]
            lora_bases.setdefault(base, {})
            lora_bases[base]['B']      = data
            lora_bases[base]['name_B'] = name

    for base, pair in lora_bases.items():
        if 'A' not in pair or 'B' not in pair:
            continue

        A = pair['A'].float()  # (old_r, in_features)
        B = pair['B'].float()  # (out_features, old_r)

        if new_r > old_r:
            # ---- Zero-padding ----
            new_A = torch.zeros(new_r, A.shape[1])
            new_A[:old_r, :] = A
            new_B = torch.zeros(B.shape[0], new_r)
            new_B[:, :old_r] = B

        else:
            # ---- SVD truncation ----
            # W = B @ A is the composite LoRA update before alpha/r scaling.
            # We want new_B @ new_A ≈ B @ A with rank = new_r.
            W = B @ A  # (out_features, in_features)
            try:
                U, S, Vh = torch.linalg.svd(W, full_matrices=False)
                # Keep top new_r singular values
                sqrt_S = torch.sqrt(S[:new_r].clamp(min=0.0))
                new_B  = U[:, :new_r] * sqrt_S          # (out, new_r)
                new_A  = Vh[:new_r, :] * sqrt_S.unsqueeze(1)  # (new_r, in)
            except RuntimeError:
                # SVD fallback: simple row/column truncation
                new_A = A[:new_r, :]
                new_B = B[:, :new_r]

        # Write into new model (match by base path + lora_A/B suffix)
        for name, param in new_model.named_parameters():
            if name.startswith(base) and 'lora_A' in name:
                param.data.copy_(new_A.to(param.dtype))
            elif name.startswith(base) and 'lora_B' in name:
                param.data.copy_(new_B.to(param.dtype))

    return new_model.to(device)


def probe_fisher(model, probe_loader, n_batches):
    """
    Cheap diagonal Fisher probe for CGRS rank-check step.
    Runs only n_batches × batch_size=4 samples.
    Returns lambda_max (scalar float).
    """
    was_training = model.training
    model.eval()
    fisher_diag  = None
    batches_done = 0

    for images, labels in probe_loader:
        if batches_done >= n_batches:
            break
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        model.zero_grad()
        outputs = model(images)
        loss    = _criterion_probe(outputs.logits, labels)
        loss.backward()
        with torch.no_grad():
            grads = torch.cat([
                p.grad.detach().pow(2).flatten()
                for p in model.parameters()
                if p.requires_grad and p.grad is not None
            ])
            fisher_diag = grads if fisher_diag is None else fisher_diag + grads
        batches_done += 1

    model.zero_grad()
    if was_training:
        model.train()

    if fisher_diag is None or batches_done == 0:
        return float('inf')   # safety fallback → triggers rank increase

    fisher_diag /= batches_done
    return float(fisher_diag.max().item())


# ---- Quick sanity-check ----
print("transition_lora_rank() and probe_fisher() ready.")
print("  transition_lora_rank: zero-pad (increase) / SVD truncation (decrease)")
print(f"  probe_fisher: {CONFIG['cgrs_probe_batches']} batches x 4 = {CONFIG['cgrs_probe_batches']*4} samples per check")

transition_lora_rank() and probe_fisher() ready.
  transition_lora_rank: zero-pad (increase) / SVD truncation (decrease)
  probe_fisher: 8 batches x 4 = 32 samples per check


In [9]:
# ----------------------------------------------------------------
# CELL 6 — CGRS training loop (fully resume-safe)
#
# SAVES after:
#   - Every rank change (mid-epoch, exact batch position saved)
#   - Every epoch end
#
# RESUMES from:
#   - Exact epoch + batch index where session ended
#   - Full rank/lambda trajectory preserved
#   - Optimizer + scheduler state restored
# ----------------------------------------------------------------
import os, json
from collections import defaultdict

CKPT_BASE = f"{PHASE3_DIR}/checkpoints"
os.makedirs(CKPT_BASE, exist_ok=True)


def _ckpt_dir(label):
    return f"{CKPT_BASE}/{label}"


def _save(label, payload, model, optimizer, scheduler):
    """Save full training state after every rank change or epoch."""
    d = _ckpt_dir(label)
    os.makedirs(d, exist_ok=True)
    torch.save(model.state_dict(),     f"{d}/model.pt")
    torch.save(optimizer.state_dict(), f"{d}/optimizer.pt")
    torch.save(scheduler.state_dict(), f"{d}/scheduler.pt")
    # Convert defaultdict keys to str for JSON
    payload_json = {**payload,
                    "rank_step_count": {str(k): v
                                        for k, v in payload["rank_step_count"].items()}}
    with open(f"{d}/state.json", "w") as f:
        json.dump(payload_json, f, indent=2)


def _load(label):
    """Load saved state. Returns (state_dict, ckpt_dir) or None."""
    d = _ckpt_dir(label)
    if not os.path.exists(f"{d}/state.json"):
        return None
    with open(f"{d}/state.json") as f:
        state = json.load(f)
    # Restore int keys for rank_step_count
    state["rank_step_count"] = defaultdict(
        int, {int(k): v for k, v in state["rank_step_count"].items()}
    )
    return state, d


def run_cgrs(tau, label):
    RANK_LIST  = CONFIG["cgrs_rank_list"]
    R_MIN      = CONFIG["cgrs_r_min"]
    R_MAX      = CONFIG["cgrs_r_max"]
    K          = CONFIG["cgrs_check_every"]
    COOLDOWN   = CONFIG["cgrs_cooldown"]
    N_PROBE    = CONFIG["cgrs_probe_batches"]
    r_init     = CONFIG["cgrs_r_init"]
    EPOCHS     = CONFIG["epochs"]
    criterion  = nn.CrossEntropyLoss()
    total_steps = EPOCHS * len(train_loader)

    print(f"\n{'='*65}")
    print(f"  {label}  |  tau={tau:.4f}  |  r_init={r_init}")
    print(f"  check_every={K} | cooldown={COOLDOWN} | probe={N_PROBE} batches")
    print(f"{'='*65}")

    # ----------------------------------------------------------------
    # RESUME or FRESH START
    # ----------------------------------------------------------------
    resume = _load(label)

    if resume is not None:
        state, ckpt_dir = resume
        start_epoch      = state["start_epoch"]
        start_batch      = state["start_batch"]   # skip batches already done
        current_r        = state["current_r"]
        global_step      = state["global_step"]
        steps_since_chg  = state["steps_since_chg"]
        rank_trajectory  = state["rank_trajectory"]
        lambda_trajectory= state["lambda_trajectory"]
        rank_changes     = state["rank_changes"]
        rank_step_count  = state["rank_step_count"]
        # Partial epoch accumulators (saved so epoch stats stay correct)
        resume_loss      = state.get("running_loss", 0.0)
        resume_correct   = state.get("correct", 0)
        resume_total     = state.get("total", 0)

        print(f"  [RESUME] epoch={start_epoch} batch={start_batch} "
              f"r={current_r} step={global_step}")

        model, _ = build_lora_model_verbose(current_r)
        model.load_state_dict(
            torch.load(f"{ckpt_dir}/model.pt", map_location=device)
        )
        optimizer = optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=CONFIG["lr_lora"], weight_decay=CONFIG["weight_decay"]
        )
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=total_steps
        )
        optimizer.load_state_dict(
            torch.load(f"{ckpt_dir}/optimizer.pt", map_location=device)
        )
        scheduler.load_state_dict(
            torch.load(f"{ckpt_dir}/scheduler.pt")
        )
    else:
        start_epoch      = 1
        start_batch      = 0
        current_r        = r_init
        global_step      = 0
        steps_since_chg  = COOLDOWN
        rank_trajectory  = []
        lambda_trajectory= []
        rank_changes     = []
        rank_step_count  = defaultdict(int)
        resume_loss      = 0.0
        resume_correct   = 0
        resume_total     = 0

        print(f"  [FRESH] Starting from epoch=1, r={r_init}")
        torch.cuda.empty_cache()
        model, _ = build_lora_model_verbose(r_init)
        optimizer = optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=CONFIG["lr_lora"], weight_decay=CONFIG["weight_decay"]
        )
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=total_steps
        )

    # ----------------------------------------------------------------
    # TRAINING LOOP
    # ----------------------------------------------------------------
    for epoch in range(start_epoch, EPOCHS + 1):
        model.train()

        # Restore partial accumulators if resuming mid-epoch,
        # otherwise reset for a fresh epoch
        if epoch == start_epoch and start_batch > 0:
            running_loss = resume_loss
            correct      = resume_correct
            total        = resume_total
        else:
            running_loss, correct, total = 0.0, 0, 0

        for batch_idx, (images, labels_b) in enumerate(train_loader):

            # Skip batches already processed before session ended
            if epoch == start_epoch and batch_idx < start_batch:
                continue

            images   = images.to(device, non_blocking=True)
            labels_b = labels_b.to(device, non_blocking=True)

            optimizer.zero_grad()
            outputs  = model(images)
            loss     = criterion(outputs.logits, labels_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                filter(lambda p: p.requires_grad, model.parameters()),
                max_norm=1.0
            )
            optimizer.step()
            scheduler.step()

            running_loss        += loss.item()
            correct             += (outputs.logits.argmax(1) == labels_b).sum().item()
            total               += labels_b.size(0)
            rank_step_count[current_r] += 1
            global_step         += 1
            steps_since_chg     += 1

            # --------------------------------------------------------
            # RANK CHECK
            # --------------------------------------------------------
            if steps_since_chg >= K:
                lmax = probe_fisher(model, curvature_probe_loader, N_PROBE)
                rank_trajectory.append((global_step, current_r))
                lambda_trajectory.append((global_step, lmax))

                rank_idx = (RANK_LIST.index(current_r) if current_r in RANK_LIST
                            else min(range(len(RANK_LIST)),
                                     key=lambda i: abs(RANK_LIST[i] - current_r)))
                new_r  = current_r
                reason = "stable"

                if lmax > tau and rank_idx < len(RANK_LIST) - 1:
                    candidate = RANK_LIST[rank_idx + 1]
                    if candidate <= R_MAX:
                        new_r  = candidate
                        reason = f"sharp (lmax={lmax:.5f} > tau={tau:.5f})"

                elif lmax < tau and rank_idx > 0:
                    candidate = RANK_LIST[rank_idx - 1]
                    if candidate >= R_MIN:
                        new_r  = candidate
                        reason = f"flat (lmax={lmax:.5f} < tau={tau:.5f})"

                if new_r != current_r:
                    direction = "UP" if new_r > current_r else "DOWN"
                    print(f"  [Step {global_step:>5}] Rank {direction}: "
                          f"{current_r} → {new_r} | {reason}")

                    rank_changes.append({
                        "step"   : global_step,
                        "old_r"  : current_r,
                        "new_r"  : new_r,
                        "lambda" : lmax,
                        "reason" : reason,
                    })

                    current_lr = float(scheduler.get_last_lr()[0])
                    model      = transition_lora_rank(model, current_r, new_r)
                    current_r  = new_r
                    remaining  = max(total_steps - global_step, 1)
                    optimizer  = optim.AdamW(
                        filter(lambda p: p.requires_grad, model.parameters()),
                        lr=current_lr, weight_decay=CONFIG["weight_decay"]
                    )
                    scheduler = optim.lr_scheduler.CosineAnnealingLR(
                        optimizer, T_max=remaining
                    )
                    torch.cuda.empty_cache()

                    # ---- SAVE immediately after every rank change ----
                    _save(label, {
                        "start_epoch"     : epoch,
                        "start_batch"     : batch_idx + 1,  # next batch to run
                        "current_r"       : current_r,
                        "global_step"     : global_step,
                        "steps_since_chg" : 0,
                        "rank_trajectory" : rank_trajectory,
                        "lambda_trajectory": lambda_trajectory,
                        "rank_changes"    : rank_changes,
                        "rank_step_count" : rank_step_count,
                        "running_loss"    : running_loss,
                        "correct"         : correct,
                        "total"           : total,
                    }, model, optimizer, scheduler)
                    print(f"  [SAVED] After rank change at step={global_step}")

                steps_since_chg = 0
            # --------------------------------------------------------

        # ---- End of epoch ----
        val_loss, val_acc = evaluate(model, val_loader)
        train_acc = 100.0 * correct / total
        print(
            f"  [{label}] Epoch {epoch}/{EPOCHS} | r={current_r} | "
            f"Train Loss: {running_loss/len(train_loader):.4f} "
            f"Train Acc: {train_acc:.2f}% | "
            f"Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.2f}%"
        )

        # ---- SAVE after every epoch ----
        _save(label, {
            "start_epoch"     : epoch + 1,  # next epoch to run
            "start_batch"     : 0,
            "current_r"       : current_r,
            "global_step"     : global_step,
            "steps_since_chg" : steps_since_chg,
            "rank_trajectory" : rank_trajectory,
            "lambda_trajectory": lambda_trajectory,
            "rank_changes"    : rank_changes,
            "rank_step_count" : rank_step_count,
            "running_loss"    : 0.0,
            "correct"         : 0,
            "total"           : 0,
        }, model, optimizer, scheduler)
        print(f"  [SAVED] Epoch {epoch} checkpoint.")

        # Reset start_batch after first (possibly partial) epoch
        start_batch = 0

    # ----------------------------------------------------------------
    # FINAL EVALUATION
    # ----------------------------------------------------------------
    test_loss, test_acc  = evaluate(model, test_loader)
    trainable_params     = sum(p.numel() for p in model.parameters()
                               if p.requires_grad)
    total_steps_done     = sum(rank_step_count.values())
    avg_rank = (sum(r * cnt for r, cnt in rank_step_count.items())
                / max(total_steps_done, 1))

    print(f"\n  [{label}] DONE -> Test Acc: {test_acc:.2f}% | "
          f"Avg rank: {avg_rank:.2f} | Changes: {len(rank_changes)}")

    del model
    torch.cuda.empty_cache()

    return {
        "test_acc"         : test_acc,
        "test_loss"        : test_loss,
        "tau"              : tau,
        "r_init"           : r_init,
        "final_rank"       : current_r,
        "avg_rank"         : avg_rank,
        "trainable_params" : trainable_params,
        "n_rank_changes"   : len(rank_changes),
        "rank_changes"     : rank_changes,
        "rank_step_count"  : dict(rank_step_count),
        "rank_trajectory"  : rank_trajectory,
        "lambda_trajectory": lambda_trajectory,
    }


print("run_cgrs() ready — saves after every rank change and every epoch.")

run_cgrs() ready — saves after every rank change and every epoch.


In [10]:
# ----------------------------------------------------------------
# CELL 7 — Load Phase 1 fixed-rank baselines
#
# Fixed LoRA r=6, r=10, r=16 are used directly from Phase 1 results.
# No retraining needed — both Phase 1 and CGRS use the same:
#   • pretrained base: google/vit-base-patch16-224
#   • dataset: CIFAR-100 (45k train, 5k val, 10k test)
#   • training budget: 3 epochs, seed=42
#   • optimizer: AdamW, lr=5e-4, cosine LR
# This makes Phase 1 results valid baselines for Phase 3 comparison.
# ----------------------------------------------------------------
phase3_results = {}   # master results dict — saved after every experiment

BASELINE_RANKS = [6, 10, 16]   # Phase 1 baselines to compare against

print("=" * 70)
print("  FIXED-RANK BASELINES (from Phase 1)")
print("=" * 70)

for r in BASELINE_RANKS:
    key_p1 = f"LoRA r={r}"
    if key_p1 in phase1_results:
        v = phase1_results[key_p1]
        phase3_results[f"Fixed r={r}"] = {
            "test_acc"         : v["test_acc"],
            "test_loss"        : v["test_loss"],
            "trainable_params" : v.get("trainable_params", "n/a"),
            "source"           : "Phase 1",
            "avg_rank"         : float(r),
        }
        print(
            f"  Fixed r={r:<5} -> Test Acc: {v['test_acc']:.2f}% | "
            f"Test Loss: {v['test_loss']:.4f}"
        )
    else:
        print(f"  Fixed r={r}: NOT FOUND in Phase 1 results.json — check Phase 1.")

# Also load full fine-tune and frozen backbone for reference
for key in ["Full fine-tune", "Frozen backbone"]:
    if key in phase1_results:
        v = phase1_results[key]
        phase3_results[key] = {
            "test_acc"  : v["test_acc"],
            "test_loss" : v["test_loss"],
            "source"    : "Phase 1",
        }
        print(f"  {key:<22} -> Test Acc: {v['test_acc']:.2f}%")

# Save initial state
with open(f"{PHASE3_DIR}/phase3_results.json", "w") as f:
    json.dump(phase3_results, f, indent=2)

print("\nBaselines loaded. phase3_results.json saved.")

  FIXED-RANK BASELINES (from Phase 1)
  Fixed r=6     -> Test Acc: 87.27% | Test Loss: 0.6846
  Fixed r=10    -> Test Acc: 89.02% | Test Loss: 0.5439
  Fixed r=16    -> Test Acc: 89.51% | Test Loss: 0.4606
  Full fine-tune         -> Test Acc: 92.87%
  Frozen backbone        -> Test Acc: 86.46%

Baselines loaded. phase3_results.json saved.


In [11]:
# CELL 8 — CGRS Run 1
tau_1   = TAU_VALUES[0]
label_1 = f"CGRS_tau{tau_1:.4f}"

if label_1 in phase3_results and "test_acc" in phase3_results[label_1]:
    print(f"Already complete: {label_1} (acc={phase3_results[label_1]['test_acc']:.2f}%)")
else:
    result_1 = run_cgrs(tau=tau_1, label=label_1)
    phase3_results[label_1] = result_1
    with open(f"{PHASE3_DIR}/phase3_results.json", "w") as f:
        json.dump(phase3_results, f, indent=2)
    with open(f"{PHASE3_DIR}/trajectory_{label_1}.json", "w") as f:
        json.dump({"rank_trajectory"  : result_1["rank_trajectory"],
                   "lambda_trajectory": result_1["lambda_trajectory"],
                   "rank_changes"     : result_1["rank_changes"],
                   "rank_step_count"  : result_1["rank_step_count"]}, f, indent=2)
    print(f"Saved → phase3_results.json")


  CGRS_tau0.0100  |  tau=0.0100  |  r_init=16
  check_every=200 | cooldown=100 | probe=8 batches
  [RESUME] epoch=4 batch=0 r=64 step=8439


[transformers] You passed `num_labels=100` which is incompatible to the `id2label` map of length `1000`.
Loading weights: 100%|██████████| 200/200 [00:00<00:00, 1098.12it/s]
[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([100])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([100, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


  LoRA r=64   | Trainable:  2,359,296 / 88,234,852 (2.674%)


/home1/hsharma9/.local/lib/python3.14/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



  [CGRS_tau0.0100] DONE -> Test Acc: 90.04% | Avg rank: 46.65 | Changes: 33
Saved → phase3_results.json


In [12]:
# CELL 9 — CGRS Run 2
tau_2   = TAU_VALUES[1]
label_2 = f"CGRS_tau{tau_2:.4f}"

if label_2 in phase3_results and "test_acc" in phase3_results[label_2]:
    print(f"Already complete: {label_2} (acc={phase3_results[label_2]['test_acc']:.2f}%)")
else:
    result_2 = run_cgrs(tau=tau_2, label=label_2)
    phase3_results[label_2] = result_2
    with open(f"{PHASE3_DIR}/phase3_results.json", "w") as f:
        json.dump(phase3_results, f, indent=2)
    with open(f"{PHASE3_DIR}/trajectory_{label_2}.json", "w") as f:
        json.dump({"rank_trajectory"  : result_2["rank_trajectory"],
                   "lambda_trajectory": result_2["lambda_trajectory"],
                   "rank_changes"     : result_2["rank_changes"],
                   "rank_step_count"  : result_2["rank_step_count"]}, f, indent=2)
    print(f"Saved → phase3_results.json")


  CGRS_tau0.0500  |  tau=0.0500  |  r_init=16
  check_every=200 | cooldown=100 | probe=8 batches
  [RESUME] epoch=3 batch=2674 r=5 step=8300


[transformers] You passed `num_labels=100` which is incompatible to the `id2label` map of length `1000`.
Loading weights: 100%|██████████| 200/200 [00:00<00:00, 9983.35it/s]
[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([100])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([100, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


  LoRA r=5    | Trainable:    184,320 / 86,059,876 (0.214%)
  [CGRS_tau0.0500] Epoch 3/3 | r=5 | Train Loss: 0.9809 Train Acc: 85.90% | Val Loss: 0.9798  Val Acc: 85.66%
  [SAVED] Epoch 3 checkpoint.

  [CGRS_tau0.0500] DONE -> Test Acc: 85.28% | Avg rank: 5.01 | Changes: 42
Saved → phase3_results.json


In [ ]:
# CELL 10 — CGRS Run 3
tau_3   = TAU_VALUES[2]
label_3 = f"CGRS_tau{tau_3:.4f}"

if label_3 in phase3_results and "test_acc" in phase3_results[label_3]:
    print(f"Already complete: {label_3} (acc={phase3_results[label_3]['test_acc']:.2f}%)")
else:
    result_3 = run_cgrs(tau=tau_3, label=label_3)
    phase3_results[label_3] = result_3
    with open(f"{PHASE3_DIR}/phase3_results.json", "w") as f:
        json.dump(phase3_results, f, indent=2)
    with open(f"{PHASE3_DIR}/trajectory_{label_3}.json", "w") as f:
        json.dump({"rank_trajectory"  : result_3["rank_trajectory"],
                   "lambda_trajectory": result_3["lambda_trajectory"],
                   "rank_changes"     : result_3["rank_changes"],
                   "rank_step_count"  : result_3["rank_step_count"]}, f, indent=2)
    print(f"Saved → phase3_results.json")


  CGRS_tau0.1000  |  tau=0.1000  |  r_init=16
  check_every=200 | cooldown=100 | probe=8 batches
  [FRESH] Starting from epoch=1, r=16


[transformers] You passed `num_labels=100` which is incompatible to the `id2label` map of length `1000`.
Loading weights: 100%|██████████| 200/200 [00:00<00:00, 3192.02it/s]
[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([100])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([100, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


  LoRA r=16   | Trainable:    589,824 / 86,465,380 (0.682%)
  [Step   100] Rank DOWN: 16 → 12 | flat (lmax=0.00174 < tau=0.10000)


[transformers] You passed `num_labels=100` which is incompatible to the `id2label` map of length `1000`.
Loading weights: 100%|██████████| 200/200 [00:00<00:00, 10320.63it/s]
[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([100])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([100, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


  [SAVED] After rank change at step=100
  [Step   300] Rank DOWN: 12 → 10 | flat (lmax=0.01317 < tau=0.10000)


...

  [SAVED] After rank change at step=500
  [Step   700] Rank DOWN: 6 → 5 | flat (lmax=0.01754 < tau=0.10000)


[transformers] You passed `num_labels=100` which is incompatible to the `id2label` map of length `1000`.
Loading weights: 100%|██████████| 200/200 [00:00<00:00, 50756.99it/s]
[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([100])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([100, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


  [SAVED] After rank change at step=700
  [Step   900] Rank DOWN: 5 → 3 | flat (lmax=0.02160 < tau=0.10000)


[transformers] You passed `num_labels=100` which is incompatible to the `id2label` map of length `1000`.
Loading weights: 100%|██████████| 200/200 [00:00<00:00, 8758.66it/s]
[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([100])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([100, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


  [SAVED] After rank change at step=900
